In [ ]:
# ============================================================
# BIASCOPE — Notebook 04 (FULL: original + fixes) — single cell
# Multilingual Dataset Preparation & EDA
# ============================================================
# NOTE: HINMIX unlabeled/synthetic hai, bias-labelled nahi.
# Bias labels baad me existing datasets + custom annotation se aayenge.
# ============================================================

import os, re, json, random, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from IPython.display import display

PROJECT_NAME = "BIASCOPE"
print(PROJECT_NAME, "— Notebook 04: Multilingual Dataset Preparation & EDA")

# ---------------- Seed ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Seed:", SEED)

# ---------------- Google Drive + paths ----------------
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MULTILINGUAL_DIR = DATA_DIR / "multilingual"
for d in (RAW_DIR, PROCESSED_DIR, MULTILINGUAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Multilingual directory:", MULTILINGUAL_DIR)

# ---------------- MBIB check + load ----------------
MBIB_TRAIN = PROCESSED_DIR / "mbib_train.csv"
MBIB_VAL = PROCESSED_DIR / "mbib_validation.csv"
MBIB_TEST = PROCESSED_DIR / "mbib_test.csv"
print("\nMBIB Train exists:", MBIB_TRAIN.exists())
print("MBIB Validation exists:", MBIB_VAL.exists())
print("MBIB Test exists:", MBIB_TEST.exists())

if MBIB_TRAIN.exists():
    mbib_train = pd.read_csv(MBIB_TRAIN)
    print("\nMBIB samples:", len(mbib_train))
    print("Columns:", mbib_train.columns.tolist())
    print("Label distribution:")
    print(mbib_train["label"].value_counts().sort_index())

# ============================================================
# HINMIX — FIX 1: sahi configs + same row indices (parallel)
# ============================================================
CONFIGS = {
    "en":      "lcsalign-en",        # English reference
    "hi":      "lcsalign-hi",        # Hindi, Devanagari
    "hicm":    "lcsalign-hicm",      # Hinglish, Devanagari code-mixed
    "hicmrom": "lcsalign-hicmrom",   # Hinglish, romanized
}
N_TOTAL = 4_200_000
HINMIX_SAMPLE_SIZE = 10_000

rng = np.random.RandomState(SEED)
idx = np.sort(rng.choice(N_TOTAL, HINMIX_SAMPLE_SIZE, replace=False))

cols = {}
for lang, cfg in CONFIGS.items():
    print(f"\nLoading {cfg} ...")
    ds = load_dataset("kartikagg98/HINMIX_hi-en", cfg, split="train")
    assert len(ds) == N_TOTAL, f"{cfg}: expected {N_TOTAL}, got {len(ds)}"
    cols[lang] = list(ds.select(idx.tolist())["text"])
    del ds

hinmix_wide = pd.DataFrame({"row_id": idx, **cols})
hinmix_df = hinmix_wide.melt(id_vars="row_id", var_name="lang", value_name="text")
hinmix_df["text"] = hinmix_df["text"].astype(str)

print("\nWide shape:", hinmix_wide.shape, "| Long shape:", hinmix_df.shape)
print("\nFirst 5 rows (wide):")
display(hinmix_wide.head())

# ============================================================
# BASIC TEXT STATISTICS (per language, raw text)
# ============================================================
hinmix_df["char_length"] = hinmix_df["text"].str.len()
hinmix_df["word_length"] = hinmix_df["text"].str.split().str.len()

print("\n=== BASIC TEXT STATISTICS (raw) ===")
display(hinmix_df.groupby("lang")[["char_length", "word_length"]]
        .agg(["mean", "median", "max"]).round(2))

# ============================================================
# SCRIPT DETECTION — FIX 4: ratio-based (raw text)
# ============================================================
DEV = r"\u0900-\u097F"

def script_ratios(t):
    dev = len(re.findall(rf"[{DEV}]", t))
    lat = len(re.findall(r"[A-Za-z]", t))
    tot = dev + lat
    return (dev / tot, lat / tot) if tot else (0.0, 0.0)

def script_type(r):
    if r.dev_ratio >= 0.15 and r.lat_ratio >= 0.15:
        return "code_mixed_devanagari"
    if r.dev_ratio >= 0.85:
        return "devanagari"
    if r.lat_ratio >= 0.85:
        return "latin"
    return "other"

def add_script_columns(df, text_col):
    r = df[text_col].map(script_ratios)
    df["dev_ratio"] = r.map(lambda x: x[0])
    df["lat_ratio"] = r.map(lambda x: x[1])
    df["script_type"] = df.apply(script_type, axis=1)
    return df

hinmix_df = add_script_columns(hinmix_df, "text")

print("\n=== LANGUAGE x SCRIPT (raw) ===")
print(pd.crosstab(hinmix_df["lang"], hinmix_df["script_type"]))

plt.figure(figsize=(10, 5))
sns.countplot(data=hinmix_df, x="script_type", hue="lang",
              order=hinmix_df["script_type"].value_counts().index)
plt.title("HINMIX Script Distribution by Language (raw)")
plt.xlabel("Script Type"); plt.ylabel("Number of Samples")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

print("\n=== CODE-MIXED EXAMPLES (hicm, raw) ===")
cm = hinmix_df[(hinmix_df["lang"] == "hicm") &
               (hinmix_df["script_type"] == "code_mixed_devanagari")]["text"]
print("Number of code-mixed (hicm) samples:", len(cm))
for i, t in enumerate(cm.head(15), 1):
    print(f"{i}. {t}")

# ============================================================
# TEXT NORMALIZATION — FIX 3: ZWJ/ZWNJ Devanagari me preserved
# ============================================================
_ZW_ALWAYS = re.compile(r"[\u200b\u200e\u200f\u202a-\u202e\u2060\ufeff]")
_ZWJ_ORPHAN = re.compile(rf"(?<![{DEV}])[\u200c\u200d]|[\u200c\u200d](?![{DEV}])")

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = _ZW_ALWAYS.sub("", text)       # remove, space se replace nahi
    text = _ZWJ_ORPHAN.sub("", text)      # ZWJ/ZWNJ sirf Devanagari ke beech rakho
    return re.sub(r"\s+", " ", text).strip()

hinmix_df["normalized_text"] = hinmix_df["text"].map(normalize_text)
hinmix_df["n_words"] = hinmix_df["normalized_text"].str.split().str.len()

print("\n=== NORMALIZATION EXAMPLES ===")
display(hinmix_df[["lang", "text", "normalized_text"]].head(8))

# ============================================================
# DUPLICATE ANALYSIS (per language) + REMOVAL (row_id level)
# ============================================================
print("\n=== DUPLICATE ANALYSIS ===")
for lang, g in hinmix_df.groupby("lang"):
    d = g["normalized_text"].duplicated().sum()
    print(f"{lang:8s} rows={len(g)}  duplicates={d}  ({d / len(g) * 100:.2f}%)")

en_rows = hinmix_df[hinmix_df["lang"] == "en"]
dup_ids = set(en_rows.loc[en_rows["normalized_text"].duplicated(), "row_id"])

before = hinmix_df["row_id"].nunique()
hinmix_clean = hinmix_df[~hinmix_df["row_id"].isin(dup_ids)].copy()
after = hinmix_clean["row_id"].nunique()
print(f"\nSentences before: {before} | after duplicate removal: {after} "
      f"| removed: {before - after}")

# ============================================================
# ARTIFACT DETECTION — FIX 2: IGNORECASE (LRB/RRB ab match honge)
# ============================================================
BROAD_ARTIFACT = re.compile(
    r"\blink\b|\blinkend\b|\bkeycap\b|\bplugins?\b|\bapp\b|<[^>]+>|\bLRB\b|\bRRB\b",
    re.IGNORECASE,
)
# Sirf inspection ke liye; in sab words ko blindly remove NAHI karte
hinmix_clean["has_artifact"] = hinmix_clean["normalized_text"].str.contains(BROAD_ARTIFACT)
print("\n=== BROAD ARTIFACT FLAG (inspection only) ===")
print(hinmix_clean.groupby("lang")["has_artifact"].sum())

print("\nBroad-artifact samples (en):")
for i, t in enumerate(hinmix_clean[(hinmix_clean["lang"] == "en") &
                                   hinmix_clean["has_artifact"]]["normalized_text"].head(20), 1):
    print(f"{i}. {t}")

STRONG_ARTIFACT = re.compile(
    r"<[^>]+>|\bkeycap\b|\blinkend\b|\b(?:begin|end) link\b|\b(?:lrb|rrb|lsb|rsb)\b",
    re.IGNORECASE,
)
hinmix_clean["strong_artifact"] = hinmix_clean["normalized_text"].str.contains(STRONG_ARTIFACT)

# koi bhi language-version kharab (artifact ya <4 words) ho to poora row_id drop
bad_ids = set(hinmix_clean.loc[
    hinmix_clean["strong_artifact"] | (hinmix_clean["n_words"] < 4), "row_id"
])
print("\nStrong-artifact / too-short sentences (row_ids):", len(bad_ids))
print("Strong-artifact examples:")
for i, t in enumerate(hinmix_clean[hinmix_clean["strong_artifact"]]["normalized_text"].head(10), 1):
    print(f"{i}. {t}")

before = hinmix_clean["row_id"].nunique()
hinmix_clean = hinmix_clean[~hinmix_clean["row_id"].isin(bad_ids)].copy()
after = hinmix_clean["row_id"].nunique()
print(f"\nSentences before artifact/short cleaning: {before} | after: {after} "
      f"| removed: {before - after}")

# ============================================================
# FINAL SCRIPT DISTRIBUTION AFTER CLEANING (normalized text)
# ============================================================
hinmix_clean = add_script_columns(hinmix_clean, "normalized_text")

print("\n=== FINAL LANGUAGE x SCRIPT (clean) ===")
print(pd.crosstab(hinmix_clean["lang"], hinmix_clean["script_type"]))
print("\nNOTE: 'hicmrom' (romanized Hinglish) 'latin' hi dikhega, English jaisa.")
print("Isliye language label 'lang' column (config provenance) se lo, script se nahi.")

for lang in CONFIGS:
    print(f"\n=== {lang} (clean samples) ===")
    for t in hinmix_clean[hinmix_clean["lang"] == lang]["normalized_text"].head(10):
        print("-", t)

# ============================================================
# SAVE
# ============================================================
out_path = MULTILINGUAL_DIR / "hinmix_clean_long.csv"
hinmix_clean.to_csv(out_path, index=False, encoding="utf-8")
print("\nSaved:", out_path, "| rows:", len(hinmix_clean),
      "| unique sentences:", hinmix_clean["row_id"].nunique())